In [ ]:
import os
import pathlib
# import json

import yaml
import numpy as np
import pandas as pd
from sklearn.manifold import TSNE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

import altair as alt
import matplotlib.pyplot as plt
import plotly.express as px

import jax
import jax.random as jr
import jax.numpy as jnp

import torch
from torch.utils.data import Subset, DataLoader

import trapallada.data
import trapallada.model
import trapallada.plot

In [ ]:
os.environ.get('CUDA_VISIBLE_DEVICES', 'No CUDA'), os.environ.get('XLA_PYTHON_CLIENT_PREALLOCATE', 'No XLA')

In [ ]:
import plotly.io as pio
pio.renderers.default = 'notebook'

In [ ]:
# input_directory =  pathlib.Path(os.environ['MY_DATA_DIR']) / 'trapallada'
input_directory =  pathlib.Path.home() / 'Sync' / 'research' / 'light curves'
assert input_directory.exists()

The *base* file name for the data

In [ ]:
# base_filename = pathlib.Path('original.pt')
base_filename = pathlib.Path('small.pt')
# base_filename = pathlib.Path('very_small.pt')

A custom *torch* `Dataset`. Returned `Tensor`s are *normalized* so that numbers are spread between $-1$ and $1$ (rather than between $0$ and $1$).

**TODO**: use *torch* machinery for normalization purposes?

`Dataset` is loaded: 18,000 gray-scale (i.e., single-channel) images. **They are normalized**.

In [ ]:
ds = trapallada.data.ImageDataset(
    input_directory / base_filename.with_stem(base_filename.stem + '_imgs'),
    input_directory / base_filename.with_stem(base_filename.stem + '_labels').with_suffix('.parquet'),
)
ds.imgs.shape

Size of `x`

In [ ]:
ds[0][1].shape

The label, `y` encodes the kind of signal

In [ ]:
ds[3][2]

Size of every image (to be used below)

In [ ]:
input_size = tuple(ds[0][1].shape[1:])
input_size

# Train-test splitting

The indexes for the split are randomly selected

In [ ]:
train_idx, validation_idx = train_test_split(np.arange(len(ds)), test_size=0.2, random_state=42, shuffle=True, stratify=ds.labels)

The actual splitting (`Subset` is a *torch* `Dataset`)

In [ ]:
train_ds = Subset(ds, train_idx)
validation_ds = Subset(ds, validation_idx)

A *PyTorch* pseudo-random numbers generator

In [ ]:
torch_rng = torch.Generator()
torch_rng.manual_seed(2147483647)

Data loaders are instantiated (same `generator` for both)

In [ ]:
# batch_size = 64
batch_size = 32
train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, generator=torch_rng)
validation_dl = DataLoader(validation_ds, batch_size=batch_size, shuffle=False, generator=torch_rng)

# Results

In [ ]:
# parameters_filename = pathlib.Path(f'weights_z={dim_z}_epochs={n_epochs}.yaml')
parameters_filename = input_directory / pathlib.Path(f'weights_z=3_epochs=5000.yaml')
assert parameters_filename.exists()

In [ ]:
with open(parameters_filename, 'r') as f:
        parameters = yaml.safe_load(f)
parameters

For the sake of readability

In [ ]:
dim_z = parameters["model"]["dim_z"]

Loading previous weights

In [ ]:
weights_filename = input_directory / pathlib.Path(f'weights_z={dim_z}_epochs={parameters["training"]["n_epochs"]}.eqx')
assert weights_filename.exists()

In [ ]:
model = trapallada.model.load_vae(weights_filename, parameters["model"]["input_size"], dim_z)
# encoder

Some reconstructions

In [ ]:
_, x, _ = next(iter(train_dl))
trapallada.plot.img_and_pred(model, x[0], dim_z)
trapallada.plot.img_and_pred(model, x[1], dim_z)
trapallada.plot.img_and_pred(model, x[3], dim_z)
trapallada.plot.img_and_pred(model, x[15], dim_z)

## Latent space

For the sake of convenience

In [ ]:
encoder, decoder = model

The `encoder` yields the mean and standard deviation

In [ ]:
encoder(x[0].numpy())

The `z` and label of every element in the training set (computed batch-wise due to memory problems). The `i`s returned by the `DataLoader` on the `Subset` are actually those in **original** `DataSet` (since underneath that's what's being used).

In [ ]:
z_s, labels, i_s = [], [], []
for i, x, l in train_dl:
    z_s.append(jax.vmap(encoder)(x.numpy()))
    labels.append(l)
    i_s.append(i)

All the values are (vertically) stacked together, and the standard deviation of `z` is dropped

In [ ]:
z_s = jnp.vstack(z_s)[:, :dim_z]
labels = torch.concat(labels)
i_s = torch.concat(i_s)
z_s.shape, labels.shape

t-SNE

In [ ]:
t_sne_embeddings = TSNE(n_components=2, perplexity=30,random_state=42).fit_transform(z_s)

The names of the *t-SNE* embedgins will be used a couple of times

In [ ]:
z_s_names = [f'z_{i}' for i in range(1, 1+dim_z)]
z_s_names

In [ ]:
# labels

In [ ]:
# ds.df['label'].cat.codes

In [ ]:
min(i_s),max(i_s)

In [ ]:
ds.df.iloc[i_s]

In [ ]:
type(i_s)
# i_s.tolist()

In [ ]:
latents_df = pd.DataFrame(np.hstack((z_s, t_sne_embeddings)), columns=z_s_names + ['tsne_1', 'tsne_2'], index=i_s.tolist())
latents_df

In [ ]:
df = pd.concat((latents_df, ds.df.iloc[i_s]), axis=1)
df

In [ ]:
alt.Chart(df.sample(n=1_000)).mark_point().encode(
    x='tsne_1:Q',
    y='tsne_2:Q',
    color='label:N',
    fill='label:N',
    size='std:Q',
    tooltip=['file name:N']
).properties(
    width=800, height=600
).interactive()

In [ ]:
# fig = px.scatter_3d(df, x='z_1', y='z_2', z='z_3', color='label', size='std')
fig = px.scatter_3d(df, x='z_1', y='z_2', z='z_3', color='label', opacity=0.5, width=800, height=800)
# fig = px.scatter_3d(df.assign(std=lambda x: np.log(x['std']) + 0.1), x='z_1', y='z_2', z='z_3', color='label', size='std')
# fig.show(renderer='iframe')
fig.show()
# fig.write_image(weights_filename.with_suffix('.pdf'))

In [ ]:
files = ['FM_s10_552.jpg', 'AM_s10_552.jpg']
# files = ['FM_s10_304.jpg', 'AM_s1_298.jpg']
# files = ['FM_s10_371', 'cLC_s2_639']
files_df = df[df['file name'].str.contains('|'.join(files), regex=True)]
print(f'distance {jnp.linalg.norm(jnp.subtract(*files_df[z_s_names].values)).item()}')
files_df

In [ ]:
# # t_sne_embeddings[files_df.index.values]
# files_z_s = z_s[files_df.index.values]
# print(f'distance {jnp.linalg.norm(jnp.subtract(*files_z_s)).item()}')
# files_z_s

In [ ]:
# images_basedir = pathlib.Path.cwd() / 'Simulations'
images_basedir = pathlib.Path.home() / 'light curves' / 'Simulations'
print(images_basedir)
assert images_basedir.exists()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
fig, axes = plt.subplots(1, 2, figsize=(20, 10))
for i_f, f in enumerate(files_df['file name']):
    image = mpimg.imread(images_basedir / f)
    axes[i_f].imshow(image)
    axes[i_f].axis('off')

In [ ]:
output_file = f'analysis_{"-".join(files)}.jpg'
fig.savefig(output_file, bbox_inches='tight')

In [ ]:
# from IPython.display import display, Markdown
# display(Markdown(f"[Open Saved Plot]({output_file})"))

In [ ]:
from IPython.display import HTML
download_link = f'<a href="{output_file}" download>Download Saved Plot</a>'
HTML(download_link)

In [ ]:
train_df = pd.read_parquet(parameters_filename.with_suffix('.parquet'))
# train_df.plot(y=('train', 'loss'), logy=True, title='Training Loss')
print(train_df.columns)
train_df

Hierarchical columns are flattened (since, apparently, `altair` doesn't support the former).

In [ ]:
train_df.columns = train_df.columns.to_flat_index().str.join('_')
train_df.columns

In [ ]:
log_likelihood_chart = alt.Chart(train_df.iloc[:].reset_index()).mark_line().encode(
    x='index:Q',
    y='train_log_likelihood:Q',
    # y=alt.Y('value:Q', scale=alt.Scale(type='log')),
).properties(
    width=600, height=600
)

kl_chart = alt.Chart(train_df.iloc[:].reset_index()).mark_line().encode(
    x='index:Q',
    y='train_KL:Q',
    color=alt.value('orange'),
    # y=alt.Y('value:Q', scale=alt.Scale(type='log')),
).properties(
    width=600, height=600
)

(log_likelihood_chart | kl_chart).interactive()

In [ ]:
print(len(df))
df[z_s_names].agg(['mean', 'std'], axis=0)